# 1.2 — Application multiplicity & the analysis grain

**Case**: Credit granting policy review for a Fintech client (Klavi).

**Question**: notebook 1.1 built the interim table at grain `uuid` + `DATA` (application date),
one row per application. Should the analysis instead work at **one application per user per
month** — a unique `(uuid, SAFRA)` key — collapsing same-month repeats with priority
`Contratado > Aprovado > Reprovado`? And does that dedup **lose feature information**?

**Answer (previewed)**: **no — keep `(uuid, DATA)`.** Same-month repeats are genuine distinct-date
re-applications; the dedup would recover ~0 snapshot information (snapshot availability tracks the
`uuid`, not the row), while `Contratado` proves at least some same-month multiples are *distinct
contracts* (different dates, sometimes different default outcomes). With no `contract_id` / loan
amount / application id to prove equivalence, keeping every application row is the conservative
choice and preserves the full through-the-door population Phase 3 needs. The only residual
trade-off — a heavy user being over-represented in univariate IV/WoE — is an **analysis-time
weighting** decision for phase 2, not a reason to mutate the clean table.

This notebook is **read-only**: it loads `data/interim/applications.parquet` and writes no data.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 50)

PROJECT_DIR = Path.cwd().parent
INTERIM_DIR = PROJECT_DIR / "data" / "interim"
APPLICATIONS_PATH = INTERIM_DIR / "applications.parquet"

df = pd.read_parquet(APPLICATIONS_PATH)

FEAT_COLS = ["score_klavi_1", "score_klavi_2", "classe_social", "status_empregaticio",
             "regiao", "Renda", "risco_aposta", "Capacidade Financeira"]
STATUS_PRIORITY = {"Contratado": 0, "Aprovado": 1, "Reprovado": 2}  # keep-preference order

print(f"applications: {len(df):,} rows x {df.shape[1]} cols  |  unique uuid: {df['uuid'].nunique():,}")
print("STATUS:", df["STATUS"].value_counts().to_dict())

applications: 220,618 rows x 17 cols  |  unique uuid: 184,009
STATUS: {'Reprovado': 119247, 'Aprovado': 77223, 'Contratado': 24148}


## 1. Multiplicity of the current grain

`(uuid, DATA)` is unique by construction (1.1's contract). The candidate grain `(uuid, SAFRA)` is
**not** — and every collision is *intra-month*: `SAFRA` equals the `DATA` month for every row, so
the repeats are the same user applying again on a different day of the same month.

In [2]:
assert not df.duplicated(["uuid", "DATA"]).any(), "current grain (uuid, DATA) must be unique"

data_month = df["DATA"].dt.year * 100 + df["DATA"].dt.month
assert (df["SAFRA"] == data_month).all(), "SAFRA must equal the DATA month"

grp_sizes = df.groupby(["uuid", "SAFRA"]).size()
rows_above_grain = len(df) - len(grp_sizes)

print(f"(uuid, SAFRA) groups           : {len(grp_sizes):,}")
print(f"groups with >1 application     : {(grp_sizes > 1).sum():,}")
print(f"rows inside multi-row groups   : {grp_sizes[grp_sizes > 1].sum():,}")
print(f"rows above the (uuid,SAFRA) grain (would be dropped by dedup): {rows_above_grain:,}")
print(f"SAFRA == DATA-month for all rows: {(df['SAFRA'] == data_month).all()}")
print("\ngroup-size distribution:")
print(grp_sizes.value_counts().sort_index().to_frame("n_groups"))

(uuid, SAFRA) groups           : 209,838
groups with >1 application     : 8,305
rows inside multi-row groups   : 19,085
rows above the (uuid,SAFRA) grain (would be dropped by dedup): 10,780
SAFRA == DATA-month for all rows: True

group-size distribution:
   n_groups
1    201533
2      6534
3      1253
4       379
5       102
6        29
7         6
8         2


**Key takeaway**: `(uuid, DATA)` is unique but `(uuid, SAFRA)` is not — **8,305 users** re-applied
within a single month (up to 8 times), **10,780 rows** above the monthly grain. Because `SAFRA`
always equals the `DATA` month, every collision is a same-month re-application on a *different day*,
never a data artifact.

## 2. What the repeats look like

If same-month repeats were mostly one funnel decision retried, status priority
(`Contratado > Aprovado > Reprovado`) would resolve them. It does not: most duplicate groups are
**single-status**, so priority can't pick a survivor and the choice would fall to an arbitrary
secondary rule.

In [3]:
dup_keys = grp_sizes[grp_sizes > 1].index
dup = df.set_index(["uuid", "SAFRA"]).loc[dup_keys]

n_status_per_group = dup.groupby(level=["uuid", "SAFRA"])["STATUS"].nunique()
status_combo = dup.groupby(level=["uuid", "SAFRA"])["STATUS"].agg(lambda s: " + ".join(sorted(set(s))))

print(f"duplicate groups                    : {len(dup_keys):,}")
print(f"  single-status (priority can't split): {(n_status_per_group == 1).sum():,}")
print(f"  multi-status  (priority applies)    : {(n_status_per_group > 1).sum():,}")
print("\nSTATUS composition of duplicate groups:")
print(status_combo.value_counts().to_frame("n_groups"))

duplicate groups                    : 8,305
  single-status (priority can't split): 7,046
  multi-status  (priority applies)    : 1,259

STATUS composition of duplicate groups:
                                   n_groups
STATUS                                     
Reprovado                              5011
Aprovado                               1955
Aprovado + Contratado                   707
Aprovado + Reprovado                    378
Contratado + Reprovado                  166
Contratado                               80
Aprovado + Contratado + Reprovado         8


**Key takeaway**: only **1,259** of 8,305 duplicate groups mix statuses — the other **7,046 are
single-status**, where the proposed priority rule gives no guidance and a survivor would be chosen
arbitrarily. Priority alone is not a sufficient dedup rule.

## 3. The `Contratado` evidence: same-month multiples are distinct contracts

`OVER30M2` (the target) exists **only** for `Contratado`. Some users have **two** `Contratado` rows
in one month. If those were the same loan double-counted they would share a date; instead they are
always on **different dates**, and in a handful the two loans even **disagree on the default
outcome** — direct proof they are separate contracts, not duplicates.

In [4]:
assert df.loc[df["STATUS"] != "Contratado", "OVER30M2"].isna().all(), \
    "OVER30M2 should be present only for Contratado"

contr = df[df["STATUS"] == "Contratado"]
n_contr_per_group = contr.groupby(["uuid", "SAFRA"]).size()
multi_keys = n_contr_per_group[n_contr_per_group > 1].index
multi = contr.set_index(["uuid", "SAFRA"]).loc[multi_keys]

distinct_dates = multi.groupby(level=["uuid", "SAFRA"])["DATA"].nunique()
day_span = multi.groupby(level=["uuid", "SAFRA"])["DATA"].agg(lambda s: (s.max() - s.min()).days)
n_labels = multi.groupby(level=["uuid", "SAFRA"])["OVER30M2"].nunique()
label_sets = multi.groupby(level=["uuid", "SAFRA"])["OVER30M2"].agg(
    lambda s: tuple(sorted(s.dropna().unique())))

print(f"total Contratado rows                       : {len(contr):,}")
print(f"(uuid, SAFRA) groups with >1 Contratado     : {len(multi_keys):,}")
print(f"  ...all on distinct dates                  : {(distinct_dates > 1).sum():,}")
print(f"  ...sharing a single date                  : {(distinct_dates == 1).sum():,}")
print(f"day-span between the two contracts (min/median/max): "
      f"{day_span.min()}/{int(day_span.median())}/{day_span.max()}")
print(f"\ngroups whose two contracts DISAGREE on OVER30M2: {(n_labels > 1).sum():,}")
print("label-set breakdown across multi-Contratado groups:")
print(label_sets.value_counts().to_frame("n_groups"))

total Contratado rows                       : 24,148
(uuid, SAFRA) groups with >1 Contratado     : 88
  ...all on distinct dates                  : 88
  ...sharing a single date                  : 0
day-span between the two contracts (min/median/max): 1/7/29

groups whose two contracts DISAGREE on OVER30M2: 7
label-set breakdown across multi-Contratado groups:
            n_groups
OVER30M2            
(0.0,)            78
(0.0, 1.0)         7
(1.0,)             3


**Key takeaway**: **88** users hold two `Contratado` in one month, **all on distinct dates**
(1–29 days apart, median 7) — never the same date. In **7** of them the two loans disagree on
`OVER30M2` (one went 30+ delinquent, the other did not). These are provably **distinct contracts**;
collapsing to one row per month would discard scarce labels and, in those 7, a real default
outcome. With no `contract_id` / amount / application id, the same caution extends to the
non-`Contratado` repeats — we cannot prove they are the "same" request either.

## 4. Would deduping lose feature information? (counterfactual)

Simulate the proposed dedup — keep by status priority, break remaining ties by **earliest**
application — and measure what the survivor loses versus the rows it would drop. The upside is
essentially nil; the downside is small but real and points the wrong way.

In [5]:
tmp = df.copy()
tmp["_prio"] = tmp["STATUS"].map(STATUS_PRIORITY)
tmp["_nfeat"] = tmp[FEAT_COLS].notna().sum(axis=1)

dup_full = tmp.set_index(["uuid", "SAFRA"]).loc[dup_keys].reset_index()
g = dup_full.groupby(["uuid", "SAFRA"])

# survivor under: status priority, then earliest application date
survivor = (dup_full.sort_values(["uuid", "SAFRA", "_prio", "DATA"])
                    .groupby(["uuid", "SAFRA"], as_index=False).first()
                    .set_index(["uuid", "SAFRA"]))

any_snapshot = g["has_snapshot"].max()
any_open_fin = g["has_open_finance"].max()
best_nfeat   = g["_nfeat"].max()

snapshot_lost = ((~survivor["has_snapshot"]) & any_snapshot.reindex(survivor.index)).sum()
open_fin_lost = ((~survivor["has_open_finance"]) & any_open_fin.reindex(survivor.index)).sum()
less_complete = (survivor["_nfeat"] < best_nfeat.reindex(survivor.index)).sum()
cells_lost    = (best_nfeat.reindex(survivor.index) - survivor["_nfeat"]).clip(lower=0).sum()

conflict_cols = ["score_klavi_2", "Renda", "classe_social", "regiao"]
conflicts = g[conflict_cols].agg(lambda s: s.dropna().nunique() > 1).any(axis=1).sum()

print("If we deduped (status priority, then earliest) across the 8,305 groups:")
print(f"  groups losing a snapshot the survivor lacked : {int(snapshot_lost)}")
print(f"  groups losing open-finance availability      : {int(open_fin_lost)}")
print(f"  groups keeping a less-complete survivor      : {int(less_complete)}  "
      f"({int(cells_lost)} feature cells)")
print(f"  groups with CONFLICTING feature values across rows: {int(conflicts)}")

If we deduped (status priority, then earliest) across the 8,305 groups:
  groups losing a snapshot the survivor lacked : 0
  groups losing open-finance availability      : 51
  groups keeping a less-complete survivor      : 143  (237 feature cells)
  groups with CONFLICTING feature values across rows: 1063


**Key takeaway**: the dedup's upside is ~zero — **0 groups** would lose a snapshot, because
snapshot availability tracks the `uuid`, not the individual row. The downside is small but real and
points the wrong way: **51** groups lose open-finance availability, **143** keep a less-complete
survivor (237 feature cells), and **1,063** groups carry *conflicting* feature values across their
daily snapshots — confirming again these are distinct events, not duplicates to be merged.

## 5. Decision & rationale

**Keep the interim grain `(uuid, DATA)` (notebook 1.1 unchanged). Do not dedup to `(uuid, SAFRA)`.**

- **No evidence of true duplication.** There is no `contract_id`, loan amount, or application id to
  prove two same-month rows are the "same" request. Deduping would impose an unverifiable
  assumption on 10,780 rows.
- **Positive evidence of distinctness.** Every multi-`Contratado` month is two contracts on
  different dates (7 with different default outcomes). It is more defensible to treat the
  non-`Contratado` repeats as distinct too than to assume the opposite.
- **Zero upside, real downside.** The dedup recovers no snapshot information (§4) yet would discard
  scarce labels, a distinct default outcome in 7 groups, and the full through-the-door population
  Phase 3's swap-set relies on.
- **The choice is reproducible either way**, but the conservative direction is to keep every
  application row and let downstream analysis decide how to weight them.

**Residual trade-off → phase 2, not here.** At `(uuid, DATA)` grain a heavy re-applier contributes
several rows, so it is slightly over-represented in univariate IV/WoE. That is an **analysis-time
weighting** decision (e.g. a one-row-per-user sensitivity check, or user-level weights when
computing IV/WoE), **not** a reason to mutate the clean interim table. Flagged for notebook 2.0.

## 6. Key takeaways

- **§1** `(uuid, DATA)` is unique; `(uuid, SAFRA)` is not — 8,305 users re-applied within a single
  month (10,780 rows above the monthly grain), and every collision is intra-month by construction
  (`SAFRA` = `DATA` month).
- **§2** Priority `Contratado > Aprovado > Reprovado` resolves only 1,259 groups; 7,046 are
  single-status, so priority alone is not a sufficient dedup rule.
- **§3** 88 users hold two `Contratado` in one month, always on distinct dates (median 7 days
  apart), 7 with conflicting `OVER30M2` — provably distinct contracts. `OVER30M2` exists only for
  `Contratado`.
- **§4** A status-priority + earliest dedup would recover **0** snapshots (availability tracks the
  `uuid`, not the row) while losing open-finance in 51 groups and completeness in 143; 1,063 groups
  hold conflicting feature values across daily snapshots.
- **Decision** Keep `(uuid, DATA)`; do not dedup. No id/amount proves equivalence, `Contratado`
  shows distinctness, dedup upside ≈ 0, and the full population is needed downstream. User-level
  over-representation in IV/WoE is a phase-2 weighting decision, not a data-cleaning one.